In [ ]:
# 14_recourse_lever_robustness.ipynb
# Revision robustness for the recourse paradox: does including household income as an
# actionable lever manufacture the perverse 'lower your income' recommendations?
# Re-run DiCE for the high-risk uninsured with income INCLUDED vs EXCLUDED from the
# actionable set. Writes t16_dice_income_lever.csv.
# Experiment D: does including income as an actionable lever manufacture the
# "perverse income-reduction" recourse? Compare DiCE with vs without income.
import warnings, os; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from differences import ATTgt
import dice_ml
PROC, TAB = "../data/processed", "../results/tables"
p=pd.read_parquet(f"{PROC}/khp_panel.parquet")
num=["age","hh_size","log_hhinc"]; cat=["female","chronic","htn","dm","disabled","HEALTH_INS","ECO1","EDU","REGION1","hh_phi_any"]
tgt="che25"; df=p[num+cat+[tgt,"PIDWON"]].dropna().copy(); df[tgt]=df[tgt].astype(int)
for c in cat: df[c]=df[c].astype(int).astype(str)
gss=GroupShuffleSplit(1,test_size=0.25,random_state=42); tr,te=next(gss.split(df,groups=df["PIDWON"]))
train,test=df.iloc[tr].copy(),df.iloc[te].copy()
pre=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),cat)],remainder="passthrough")
clf=Pipeline([("pre",pre),("gb",HistGradientBoostingClassifier(max_iter=350,learning_rate=0.06,max_depth=4,l2_regularization=1.0,random_state=0))])
clf.fit(train[num+cat],train[tgt])
print("AUC=%.3f"%roc_auc_score(test[tgt],clf.predict_proba(test[num+cat])[:,1]))

# associational vs design-based contrast (reproduce the paradox)
A=test.copy();A["hh_phi_any"]="0";B=test.copy();B["hh_phi_any"]="1"
assoc=(clf.predict_proba(B[num+cat])[:,1]-clf.predict_proba(A[num+cat])[:,1]).mean()
d2=p[["PIDWON","year","cohort","che25","base_phi"]].dropna(subset=["che25"]).copy()
d2["cohort"]=d2["cohort"].replace(0,np.nan);d2=d2.set_index(["PIDWON","year"])
att=ATTgt(data=d2,cohort_column="cohort",base_period="universal")
att.fit(formula="che25",est_method="dr",control_group="not_yet_treated",sample_split_column="base_phi",progress_bar=False)
s=att.aggregate("simple");s.columns=[str(c) for c in s.columns];causal=float(s.iloc[1,0])-float(s.iloc[0,0])
print("PHI effect on CHE25 -- associational: %+.4f | design-based: %+.4f"%(assoc,causal))

# DiCE: income INCLUDED vs EXCLUDED
dd=dice_ml.Data(dataframe=train[num+cat+[tgt]],continuous_features=num,outcome_name=tgt)
m=dice_ml.Model(model=clf,backend="sklearn")
unins=test[test["hh_phi_any"]=="0"].copy(); unins["risk"]=clf.predict_proba(unins[num+cat])[:,1]
hi=unins.sort_values("risk",ascending=False).head(20)[num+cat].reset_index(drop=True)
exp=dice_ml.Dice(dd,m,method="genetic")
def run(vary,label):
    cf=exp.generate_counterfactuals(hi,total_CFs=3,desired_class=0,features_to_vary=vary)
    ncf=phi=inc_down=welfare=valid=0
    for i in range(len(hi)):
        c=cf.cf_examples_list[i].final_cfs_df
        if c is None or len(c)==0: continue
        valid+=1; o=hi.iloc[i]
        for j in range(len(c)):
            ncf+=1; row=c.iloc[j]
            if str(row["hh_phi_any"])!=str(o["hh_phi_any"]): phi+=1
            if "log_hhinc" in vary and float(row["log_hhinc"])<float(o["log_hhinc"])-1e-6: inc_down+=1
            if str(row["HEALTH_INS"]) in ("5","6") and str(o["HEALTH_INS"]) not in ("5","6"): welfare+=1
    f=lambda x:100*x/ncf if ncf else 0
    return {"setting":label,"valid_of20":valid,"n_cf":ncf,"phi_reco_pct":f(phi),"income_down_pct":f(inc_down),"welfare_switch_pct":f(welfare)}
rows=[run(["hh_phi_any","HEALTH_INS","ECO1","log_hhinc"],"income INCLUDED (baseline)"),
      run(["hh_phi_any","HEALTH_INS","ECO1"],"income EXCLUDED")]
out=pd.DataFrame(rows); out.to_csv(f"{TAB}/t16_dice_income_lever.csv",index=False)
print("\n=== DiCE recourse: income lever included vs excluded (top-20 high-risk uninsured) ===")
print(out.round(1).to_string(index=False))
